In [3]:
vivado -mode tcl

SyntaxError: invalid syntax (3172630247.py, line 1)

In [4]:
import numpy as np
from onnx import helper

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.base import Transformation
from qonnx.transformation.general import GiveUniqueNodeNames, GiveReadableTensorNames
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes


def _get_perm(node):
    for a in node.attribute:
        if a.name == "perm":
            return list(a.ints)
    return None


def _find_predecessor_by_op(model, node, op_type):
    preds = model.find_direct_predecessors(node)
    if preds is None:
        return None
    for p in preds:
        if p.op_type == op_type:
            return p
    return None


def _find_successor_by_op(model, node, op_type):
    succs = model.find_direct_successors(node)
    if succs is None:
        return None
    for s in succs:
        if s.op_type == op_type:
            return s
    return None


def _find_transpose_predecessor_with_perm(model, node, perm_target):
    preds = model.find_direct_predecessors(node)
    if preds is None:
        return None
    for p in preds:
        if p.op_type != "Transpose":
            continue
        perm = _get_perm(p)
        if perm == perm_target:
            return p
    return None


def _find_transpose_successor_with_perm(model, node, perm_target):
    succs = model.find_direct_successors(node)
    if succs is None:
        return None
    for s in succs:
        if s.op_type != "Transpose":
            continue
        perm = _get_perm(s)
        if perm == perm_target:
            return s
    return None


def _get_const_tensor(model, name):
    # 1) initializer
    try:
        arr = model.get_initializer(name)
        if arr is not None:
            return arr
    except Exception:
        pass

    # 2) Constant node
    prod = model.find_producer(name)
    if prod is None or prod.op_type != "Constant":
        return None

    for a in prod.attribute:
        if a.name == "value":
            from onnx import numpy_helper
            return numpy_helper.to_array(a.t)

    return None


def _get_pad_mode(pad_node):
    # 默认 constant
    mode = "constant"
    for a in pad_node.attribute:
        if a.name == "mode":
            try:
                mode = a.s.decode("utf-8")
            except Exception:
                mode = str(a.s)
    return mode


def _get_pad_value(model, pad_node):
    # 新 opset: input[2]
    if len(pad_node.input) >= 3:
        cval = _get_const_tensor(model, pad_node.input[2])
        if cval is not None:
            cval = np.asarray(cval).reshape(-1)
            if len(cval) == 0:
                return 0.0
            return float(cval[0])

    # 老 opset: attribute "value"
    for a in pad_node.attribute:
        if a.name == "value":
            return float(a.f)

    return 0.0


def _get_pad_vector(model, pad_node):
    # 新 opset: input[1]
    if len(pad_node.input) >= 2:
        pads = _get_const_tensor(model, pad_node.input[1])
        if pads is not None:
            return [int(x) for x in np.asarray(pads).reshape(-1).tolist()]

    # 老 opset: attribute "pads"
    for a in pad_node.attribute:
        if a.name == "pads":
            return [int(x) for x in a.ints]

    return None


def _extract_nchw_pad_info(model, pad_node):
    """
    NCHW 4D pad:
    [N_begin, C_begin, H_begin, W_begin, N_end, C_end, H_end, W_end]
    返回 [Top, Left, Bottom, Right]
    """
    pads = _get_pad_vector(model, pad_node)
    if pads is None or len(pads) != 8:
        return None

    n_beg, c_beg, h_beg, w_beg, n_end, c_end, h_end, w_end = pads

    if n_beg != 0 or c_beg != 0 or n_end != 0 or c_end != 0:
        return None

    return [h_beg, w_beg, h_end, w_end]


class LowerPadSandwichToFMPadding(Transformation):
    """
    Match:
        Transpose [0,3,1,2] -> Pad(NCHW, constant zero) -> Transpose [0,2,3,1]
    Replace with:
        FMPadding in NHWC domain
    """

    def apply(self, model: ModelWrapper):
        graph = model.graph
        modified = False

        for pad in list(graph.node):
            if pad.op_type != "Pad":
                continue

            # 只处理 constant zero pad
            if _get_pad_mode(pad) != "constant":
                continue
            if _get_pad_value(model, pad) != 0.0:
                continue

            t1 = _find_transpose_predecessor_with_perm(model, pad, [0, 3, 1, 2])
            if t1 is None:
                continue

            t2 = _find_transpose_successor_with_perm(model, pad, [0, 2, 3, 1])
            if t2 is None:
                continue

            pad_tlbr = _extract_nchw_pad_info(model, pad)
            if pad_tlbr is None:
                continue

            inp_name = t1.input[0]
            out_name = t2.output[0]

            ishape = model.get_tensor_shape(inp_name)
            if ishape is None or len(ishape) != 4:
                continue

            n, h, w, c = [int(x) for x in ishape]
            pad_t, pad_l, pad_b, pad_r = pad_tlbr
            oshape = [n, h + pad_t + pad_b, w + pad_l + pad_r, c]

            in_dt = model.get_tensor_datatype(inp_name)
            in_dt_name = in_dt.name if hasattr(in_dt, "name") else str(in_dt)

            print(
                "Matched pad sandwich:",
                t1.name if t1.name else "<unnamed>",
                "->",
                pad.name if pad.name else "<unnamed>",
                "->",
                t2.name if t2.name else "<unnamed>",
                "| pad tlbr =",
                [pad_t, pad_l, pad_b, pad_r],
                "| input shape =",
                ishape,
            )

            new_node = helper.make_node(
                "FMPadding",
                inputs=[inp_name],
                outputs=[out_name],
                domain="finn.custom_op.fpgadataflow",
                backend="fpgadataflow",
                ImgDim=[int(h), int(w)],
                Padding=[int(pad_t), int(pad_l), int(pad_b), int(pad_r)],
                NumChannels=int(c),
                SIMD=int(c),
                inputDataType=in_dt_name,
                name="FMPadding_" + (pad.name if pad.name else "auto"),
            )

            insert_idx = list(graph.node).index(t1)
            graph.node.insert(insert_idx, new_node)

            graph.node.remove(t1)
            graph.node.remove(pad)
            graph.node.remove(t2)

            model.set_tensor_shape(out_name, oshape)
            model.set_tensor_datatype(out_name, in_dt)

            modified = True

        return (model, modified)


if __name__ == "__main__":
    in_path = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup.onnx"
    out_path = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad.onnx"

    model = ModelWrapper(in_path)
    model = model.transform(LowerPadSandwichToFMPadding())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    model.save(out_path)

    print("Saved:", out_path)

Matched pad sandwich: Transpose_3 -> Pad_1 -> Transpose_4 | pad tlbr = [0, 2, 0, 0] | input shape = [1, 1, 140, 16]
Matched pad sandwich: Transpose_7 -> Pad_2 -> Transpose_9 | pad tlbr = [0, 4, 0, 0] | input shape = [1, 1, 140, 16]
Matched pad sandwich: Transpose_11 -> Pad_3 -> Transpose_12 | pad tlbr = [0, 4, 0, 0] | input shape = [1, 1, 140, 16]
Matched pad sandwich: <unnamed> -> Pad_4 -> Transpose_17 | pad tlbr = [0, 8, 0, 0] | input shape = [1, 1, 140, 16]
Matched pad sandwich: Transpose_19 -> Pad_5 -> Transpose_20 | pad tlbr = [0, 8, 0, 0] | input shape = [1, 1, 140, 32]
Saved: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad.onnx


In [6]:
from qonnx.core.modelwrapper import ModelWrapper

m = ModelWrapper("/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad.onnx")

targets = {"Transpose_0", "Pad_0", "Transpose_1", "DuplicateStreams_0", "ConvolutionInputGenerator_0", "MVAU_0"}

for n in m.graph.node:
    name = n.name if n.name else "<unnamed>"
    if name not in targets:
        continue

    print("\n==============================")
    print("node:", name, "|", n.op_type)
    print(" inputs :", list(n.input))
    print(" outputs:", list(n.output))

    for a in n.attribute:
        if a.name == "perm":
            print(" perm:", list(a.ints))
        elif a.name == "pads":
            print(" pads attr:", list(a.ints))
        elif a.name == "mode":
            try:
                print(" mode:", a.s.decode("utf-8"))
            except Exception:
                print(" mode raw:", a.s)
        elif a.name == "value":
            try:
                print(" value float:", a.f)
            except Exception:
                print(" value attr present")

    preds = m.find_direct_predecessors(n)
    succs = m.find_direct_successors(n)

    print(" preds :", [] if preds is None else [(p.name if p.name else '<unnamed>', p.op_type) for p in preds])
    print(" succs :", [] if succs is None else [(s.name if s.name else '<unnamed>', s.op_type) for s in succs])

    for x in n.input:
        try:
            print("  in shape :", x, m.get_tensor_shape(x))
        except Exception:
            pass

    for x in n.output:
        try:
            print("  out shape:", x, m.get_tensor_shape(x))
        except Exception:
            pass


node: DuplicateStreams_0 | DuplicateStreams
 inputs : ['MultiThreshold_0_out0']
 outputs: ['DuplicateStreams_0_out0', 'DuplicateStreams_0_out1']
 preds : [('MultiThreshold_0', 'MultiThreshold')]
 succs : [('Transpose_0', 'Transpose'), ('Pad_0', 'Pad')]
  in shape : MultiThreshold_0_out0 [1, 1, 1, 140]
  out shape: DuplicateStreams_0_out0 [1, 1, 1, 140]
  out shape: DuplicateStreams_0_out1 [1, 1, 1, 140]

node: Transpose_0 | Transpose
 inputs : ['DuplicateStreams_0_out0']
 outputs: ['Transpose_0_out0']
 perm: [0, 2, 3, 1]
 preds : [('DuplicateStreams_0', 'DuplicateStreams')]
 succs : [('MVAU_0', 'MVAU')]
  in shape : DuplicateStreams_0_out0 [1, 1, 1, 140]
  out shape: Transpose_0_out0 [1, 1, 140, 1]

node: Pad_0 | Pad
 inputs : ['DuplicateStreams_0_out1', 'Pad_0_param0', 'Pad_0_param1']
 outputs: ['Pad_0_out0']
 mode: constant
 preds : [('DuplicateStreams_0', 'DuplicateStreams')]
 succs : [('Transpose_1', 'Transpose')]
  in shape : DuplicateStreams_0_out1 [1, 1, 1, 140]
  in shape : Pa

In [13]:
import numpy as np
from onnx import helper

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.base import Transformation
from qonnx.transformation.general import GiveUniqueNodeNames, GiveReadableTensorNames
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.infer_datatypes import InferDataTypes


def _get_perm(node):
    for a in node.attribute:
        if a.name == "perm":
            return list(a.ints)
    return None


def _find_predecessor_by_op(model, node, op_type):
    preds = model.find_direct_predecessors(node)
    if preds is None:
        return None
    for p in preds:
        if p.op_type == op_type:
            return p
    return None


def _find_successor_by_op(model, node, op_type):
    succs = model.find_direct_successors(node)
    if succs is None:
        return None
    for s in succs:
        if s.op_type == op_type:
            return s
    return None


def _get_const_tensor(model, name):
    try:
        arr = model.get_initializer(name)
        if arr is not None:
            return arr
    except Exception:
        pass

    prod = model.find_producer(name)
    if prod is None or prod.op_type != "Constant":
        return None

    for a in prod.attribute:
        if a.name == "value":
            from onnx import numpy_helper
            return numpy_helper.to_array(a.t)

    return None


def _get_pad_mode(pad_node):
    mode = "constant"
    for a in pad_node.attribute:
        if a.name == "mode":
            try:
                mode = a.s.decode("utf-8")
            except Exception:
                mode = str(a.s)
    return mode


def _get_pad_value(model, pad_node):
    if len(pad_node.input) >= 3:
        cval = _get_const_tensor(model, pad_node.input[2])
        if cval is not None:
            cval = np.asarray(cval).reshape(-1)
            if len(cval) == 0:
                return 0.0
            return float(cval[0])

    for a in pad_node.attribute:
        if a.name == "value":
            return float(a.f)

    return 0.0


def _get_pad_vector(model, pad_node):
    if len(pad_node.input) >= 2:
        pads = _get_const_tensor(model, pad_node.input[1])
        if pads is not None:
            return [int(x) for x in np.asarray(pads).reshape(-1).tolist()]

    for a in pad_node.attribute:
        if a.name == "pads":
            return [int(x) for x in a.ints]

    return None


def _extract_nchw_pad_info(model, pad_node):
    pads = _get_pad_vector(model, pad_node)
    if pads is None or len(pads) != 8:
        return None

    n_beg, c_beg, h_beg, w_beg, n_end, c_end, h_end, w_end = pads

    if n_beg != 0 or c_beg != 0 or n_end != 0 or c_end != 0:
        return None

    return [h_beg, w_beg, h_end, w_end]  # [Top, Left, Bottom, Right]


def _replace_input(node, old_name, new_name):
    for i, x in enumerate(node.input):
        if x == old_name:
            node.input[i] = new_name


class LowerInputForkPadToFMPadding(Transformation):
    """
    Match:
        DuplicateStreams
          ├─ Transpose([0,2,3,1]) -> MVAU
          └─ Pad(NCHW, const 0) -> Transpose([0,2,3,1]) -> ConvolutionInputGenerator

    Rewrite to:
        Transpose_pre([0,2,3,1]) -> DuplicateStreams
          ├─ -> MVAU
          └─ -> FMPadding -> ConvolutionInputGenerator
    """

    def apply(self, model: ModelWrapper):
        graph = model.graph
        modified = False

        for pad in list(graph.node):
            if pad.op_type != "Pad":
                continue

            if _get_pad_mode(pad) != "constant":
                continue
            if _get_pad_value(model, pad) != 0.0:
                continue

            t_pad = _find_successor_by_op(model, pad, "Transpose")
            if t_pad is None or _get_perm(t_pad) != [0, 2, 3, 1]:
                continue

            dup = _find_predecessor_by_op(model, pad, "DuplicateStreams")
            if dup is None:
                continue

            t_direct = None
            succs = model.find_direct_successors(dup)
            if succs is None:
                continue

            for s in succs:
                if s == pad:
                    continue
                if s.op_type == "Transpose" and _get_perm(s) == [0, 2, 3, 1]:
                    t_direct = s
                    break

            if t_direct is None:
                continue

            mvau0 = _find_successor_by_op(model, t_direct, "MVAU")
            cig0 = _find_successor_by_op(model, t_pad, "ConvolutionInputGenerator")
            if mvau0 is None or cig0 is None:
                continue

            old_dup_in = dup.input[0]
            ishape_nchw = model.get_tensor_shape(old_dup_in)
            if ishape_nchw is None or len(ishape_nchw) != 4:
                continue

            n, c, h, w = [int(x) for x in ishape_nchw]
            ishape_nhwc = [n, h, w, c]

            pad_tlbr = _extract_nchw_pad_info(model, pad)
            if pad_tlbr is None:
                continue
            pad_t, pad_l, pad_b, pad_r = pad_tlbr

            dup_dt = model.get_tensor_datatype(old_dup_in)
            dup_dt_name = dup_dt.name if hasattr(dup_dt, "name") else str(dup_dt)

            print(
                "Matched input fork pattern:",
                dup.name if dup.name else "<unnamed>",
                "| direct branch =", t_direct.name if t_direct.name else "<unnamed>",
                "| pad branch =", pad.name if pad.name else "<unnamed>",
                "->",
                t_pad.name if t_pad.name else "<unnamed>",
                "| old dup in =", ishape_nchw,
                "| new dup in =", ishape_nhwc,
            )

            # 1) 在 DuplicateStreams 前插入统一的 NCHW->NHWC transpose
            nhwc_in_name = old_dup_in + "_nhwc"
            t_pre = helper.make_node(
                "Transpose",
                inputs=[old_dup_in],
                outputs=[nhwc_in_name],
                perm=[0, 2, 3, 1],
                name="Transpose_pre_" + (dup.name if dup.name else "dup"),
            )

            dup_insert_idx = list(graph.node).index(dup)
            graph.node.insert(dup_insert_idx, t_pre)

            model.set_tensor_shape(nhwc_in_name, ishape_nhwc)
            model.set_tensor_datatype(nhwc_in_name, dup_dt)

            # 2) DuplicateStreams 输入改成 NHWC
            dup.input[0] = nhwc_in_name

            # 3) DuplicateStreams 自己的属性必须同步改
            dup_inst = getCustomOp(dup)
            dup_inst.set_nodeattr("NumChannels", int(c))
            dup_inst.set_nodeattr("numInputVectors", [int(n), int(h), int(w)])
            dup_inst.set_nodeattr("PE", 1)

            try:
                dup_inst.set_nodeattr("inputDataType", dup_dt_name)
            except Exception:
                pass

            # 输出 tensor metadata 也改成 NHWC
            for out_name in dup.output:
                model.set_tensor_shape(out_name, ishape_nhwc)
                model.set_tensor_datatype(out_name, dup_dt)

            # 4) 直连分支：删掉原来的 transpose，让 MVAU 直接吃 DuplicateStreams 输出
            _replace_input(mvau0, t_direct.output[0], t_direct.input[0])

            # 5) Pad 分支：Pad + Transpose 合成 FMPadding
            fmpad_out = t_pad.output[0]
            fmpad_shape = [n, h + pad_t + pad_b, w + pad_l + pad_r, c]

            fmpad = helper.make_node(
                "FMPadding",
                inputs=[pad.input[0]],
                outputs=[fmpad_out],
                domain="finn.custom_op.fpgadataflow",
                backend="fpgadataflow",
                ImgDim=[int(h), int(w)],
                Padding=[int(pad_t), int(pad_l), int(pad_b), int(pad_r)],
                NumChannels=int(c),
                SIMD=int(c),
                inputDataType=dup_dt_name,
                name="FMPadding_input_" + (pad.name if pad.name else "auto"),
            )

            pad_insert_idx = list(graph.node).index(pad)
            graph.node.insert(pad_insert_idx, fmpad)

            model.set_tensor_shape(fmpad_out, fmpad_shape)
            model.set_tensor_datatype(fmpad_out, dup_dt)

            # 6) 删除旧节点
            graph.node.remove(t_direct)
            graph.node.remove(pad)
            graph.node.remove(t_pad)

            modified = True

        return (model, modified)


if __name__ == "__main__":
    in_path = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad.onnx"
    out_path = "/home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx"

    model = ModelWrapper(in_path)
    model = model.transform(LowerInputForkPadToFMPadding())
    model = model.transform(GiveUniqueNodeNames())
    model = model.transform(GiveReadableTensorNames())
    model = model.transform(InferShapes())
    model = model.transform(InferDataTypes())
    model.save(out_path)

    print("Saved:", out_path)

Matched input fork pattern: DuplicateStreams_0 | direct branch = Transpose_0 | pad branch = Pad_0 -> Transpose_1 | old dup in = [1, 1, 1, 140] | new dup in = [1, 1, 140, 1]
Saved: /home/slowman/Desktop/project/Thesis/finn/notebooks/pytorch-tcn/finn_build_pynqz2/intermediate_models/step_after_convert_to_hw_cleanup_fmpad_inputfix.onnx
